In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import sys

#math and array operations
import numpy as np
import math

#plotting
import matplotlib
matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#data classes
import xarray as xr
import pickle 

#file classes
import glob

#loading bar
from tqdm import tqdm

#datetime 
from datetime import datetime

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "MPAS_Model_Data", "AdditionalVariables")
dataType = "AdditionalVariablesAnimation"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_PlottingModelData import FigurePlotting_Class

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class,DataOperator_Class

In [ ]:
#Setup

# Region = "TRACER"; Case = "WET"; spinup_hours = "0"
# Region = "TRACER"; Case = "DIURNAL"; spinup_hours = "-5"

# Region = "PRECIP"; Case = "WET"; spinup_hours = "12"
# Region = "PRECIP"; Case = "DIURNAL"; spinup_hours = "12"

# Region = "Hawaii"; Case = "WET"; spinup_hours = "12"
Region = "Hawaii"; Case = "TRADES"; spinup_hours = "24"

In [ ]:
#Load Model Directory Class
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData_NSSL = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

RunType = (Region,Case,"TEMPO",spinup_hours)
ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

In [ ]:
#Importing PlottingModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_RadarDataPlotting import RadarPlotting_Class

In [ ]:
###############
#JOB ARRAY SETUP

In [ ]:
#Importing PlottingModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_JobArray import JobArray_Class

In [ ]:
#JOB ARRAY SETUP
UsingJobArray=False

def GetNumJobs():
    num_jobs=20
    return num_jobs

num_jobs = GetNumJobs()
JobArray = JobArray_Class(total_elements=ModelData_NSSL.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = JobArray.start_job; end_job = JobArray.end_job

def GetNumElements():
    num_elements = np.arange(ModelData_NSSL.Ntime)[start_job:end_job].tolist()
    return num_elements
num_elements = GetNumElements()

In [ ]:
###############
#FUNCTIONS

In [ ]:
def LoadAdditionalVariableData(ModelData, variableName,t,printout=True):

    codeType = os.path.join("DataAnalysis", "MPAS_Model_Data", "AdditionalVariables")
    dataType = "DensityPotentialTemperature"
    outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)

    outputFolder = f"{ModelData.region}_{ModelData.case}_{ModelData.mpType}_spinup{ModelData.spinup_hours}hrs"
    outputFolderPath = os.path.join(outputDirectory,outputFolder,variableName)
    os.makedirs(outputFolderPath, exist_ok=True)
    
    outputFile = f"{variableName}_{ModelData.timeStrings[t]}.nc"
    outputFilePath = os.path.join(outputFolderPath,outputFile)
        
    outputData = xr.open_dataset(outputFilePath)

    if printout == True:
        print(f"Loaded from {outputFilePath}","\n")

    outputData = outputData["__xarray_dataarray_variable__"]
    return outputData

In [ ]:
def BuildVariableDictionary(ModelData, varNames, variableSubset_Dictionary,
                            lat,lon,climDictionary):
    variableDictionary = {}
    for varName in varNames:
        clim = climDictionary[varName]

        #Setting up Units and Multiplier
        if varName in ["Theta_rho"]:
            units = r"$(K)$"
            cmap = "turbo"
        else:
            units = ""
            cmap = "viridis"

        # Store in dictionary
        variableDictionary[varName] = {
            "data": variableSubset_Dictionary[varName],
            "lat": lat,
            "lon": lon,
            "units": units,
            "multiplier": 1,
            # "outputFilePath": outputFilePath,
            "clim": clim,
            "center_colorbar": False,
            "cmap": cmap
        }
        
    return variableDictionary

In [ ]:
# ============================================================
# DataOperator_Class
# ============================================================

import math
import xarray as xr
import numpy as np

class DataOperator_Class:

    @staticmethod
    def LatLonBoundingBox_Center(region="TRACER"): 
        if region == "TRACER":
            (latCenter, lonCenter) = 29.67, -95.059
        elif region == "PRECIP":
            (latCenter, lonCenter) = 24.82, 120.91
        elif region == "Hawaii":
            (latCenter, lonCenter) = 21.133, -157.180 
        return latCenter,lonCenter

    @staticmethod
    def LatLonBoundingBox_Calculation(latCenter, lonCenter, radius_km=500): 
        # Earth radius in km
        R = 6371.0
    
        # Convert degrees to radians
        latRadians = math.radians(latCenter)
    
        # Calculate degree offsets
        dLat = (radius_km / R) * (180.0 / math.pi)
        dLon = (radius_km / (R * math.cos(latRadians))) * (180.0 / math.pi)
    
        # Bounding box
        latMin = latCenter - dLat
        latMax = latCenter + dLat
        lonMin = lonCenter - dLon
        lonMax = lonCenter + dLon
    
        latBounds = (latMin, latMax)
        lonBounds = (lonMin, lonMax)
        return latBounds, lonBounds

    @staticmethod
    def LatLonBoundingBox_Subset(data, latBounds, lonBounds): 
        """
        Subset a structured (lat-lon) xarray DataArray or Dataset
        to a given latitude/longitude bounding box.
        """
        # Determine coordinate names (support latitude/lat, longitude/lon)
        lat_name = "latitude" #if "latitude" in data.coords else "lat"
        lon_name = "longitude" #if "longitude" in data.coords else "lon"
    
        # Handle reversed latitude (if decreasing)
        lat_vals = data[lat_name].values
        if lat_vals[0] > lat_vals[-1]:
            lat_slice = slice(latBounds[1], latBounds[0])
        else:
            lat_slice = slice(latBounds[0], latBounds[1])
    
        lon_slice = slice(lonBounds[0], lonBounds[1])
    
        # Perform subset
        dataSubset = data.sel({lat_name: lat_slice, lon_name: lon_slice})
    
        # Extract matching lat/lon arrays
        lat = dataSubset[lat_name].values
        lon = dataSubset[lon_name].values
    
        # print(f"Subset region: lat={latBounds}, lon={lonBounds}")
        # print(f"Subset shape: {dataSubset[lat_name].shape} × {variableSubset[lon_name].shape}")
    
        return dataSubset, lat, lon

    @staticmethod
    def GetData_Variable(ModelData, data, data_diag, data_static, varName):
        if varName in ModelData.unitsDictionary:
            return data[varName]
        elif varName in ModelData.unitsDictionary_diag:
            return data_diag[varName]
        elif varName == "greenfrac":
            return data_static[varName].isel(nMonths=6)

    @staticmethod
    def GetVariable_Subset(ModelData, data, data_diag, data_static, varName):  
        variable = DataOperator_Class.GetData_Variable(ModelData, data, data_diag, data_static, varName)
        [latCenter,lonCenter] = DataOperator_Class.LatLonBoundingBox_Center(region=ModelData.region)
        [latBounds, lonBounds] = DataOperator_Class.LatLonBoundingBox_Calculation(latCenter, lonCenter, radius_km=500)
        variableSubset, lat, lon = DataOperator_Class.LatLonBoundingBox_Subset(variable,latBounds, lonBounds)
    
        # Lon, Lat = np.meshgrid(lon, lat) #not actually needed to plot
        return variableSubset, lat, lon

    @staticmethod
    def GetVariable_Subset2(ModelData, variable):
        [latCenter,lonCenter] = DataOperator_Class.LatLonBoundingBox_Center(region=ModelData.region)
        [latBounds, lonBounds] = DataOperator_Class.LatLonBoundingBox_Calculation(latCenter, lonCenter, radius_km=500)
        variableSubset, lat, lon = DataOperator_Class.LatLonBoundingBox_Subset(variable,latBounds, lonBounds)

        return variableSubset, lat, lon

    @staticmethod
    def GetData_Subset(ModelData,t):  
        data = ModelData.GetDataTimestep(t,printout=False)
        data_diag = ModelData.GetDataTimestep_diag(t,printout=False)
    
        [latCenter,lonCenter] = DataOperator_Class.LatLonBoundingBox_Center(region=ModelData.region)
        [latBounds, lonBounds] = DataOperator_Class.LatLonBoundingBox_Calculation(latCenter, lonCenter, radius_km=500)
        dataSubset, lat, lon = DataOperator_Class.LatLonBoundingBox_Subset(data,latBounds, lonBounds)
        dataSubset_diag, _, _ = DataOperator_Class.LatLonBoundingBox_Subset(data_diag,latBounds, lonBounds)
        dataSubset_static, _, _ = DataOperator_Class.LatLonBoundingBox_Subset(ModelData.staticData,latBounds, lonBounds)
    
        # Lon, Lat = np.meshgrid(lon, lat) #not actually needed to plot
        return dataSubset, dataSubset_diag, dataSubset_static, lat, lon, data, data_diag

    @staticmethod
    def GetOutputFilePath(ModelData, DirectoryManager, outputDirectory, fileName):
        folderName = f"{ModelData.region}_{ModelData.case}_{ModelData.mpType}_{ModelData.spinup_hours}hrs"    
        filePath = DirectoryManager.GetOutputFile(outputDirectory, folderName, fileName)
        return filePath

In [ ]:
def GetCLims_Average(ModelData, varNames, method="mean",
                     lower_pct=5, upper_pct=95):
    """
    Compute average or percentile-based (vmin, vmax) across all timesteps.
    Also stores the full list of per-timestep min/max values.
    Returns:
        climDictionary (summary dict[varName] = (vmin, vmax))
        full_climDictionary (detailed dict[varName] = {"vmins": [...], "vmaxs": [...]})
    """
    import numpy as np
    from tqdm import tqdm

    if isinstance(varNames, str):
        varNames = [varNames]

    # store lists of per-timestep values
    full_climDictionary = {v: {"vmins": [], "vmaxs": []} for v in varNames}

    for t in tqdm(range(ModelData.Ntime), desc="Processing timesteps"):
        for varName in varNames:

            variable = LoadAdditionalVariableData(ModelData, varName,t,printout=True)
            variableSubset, _, _ = DataOperator_Class.GetVariable_Subset2(ModelData, variable)
            
            vmin = np.nanmin(variableSubset)
            vmax = np.nanmax(variableSubset)
            full_climDictionary[varName]["vmins"].append(vmin)
            full_climDictionary[varName]["vmaxs"].append(vmax)

    # summarize
    climDictionary = {}
    for varName in varNames:
        vmins = np.array(full_climDictionary[varName]["vmins"])
        vmaxs = np.array(full_climDictionary[varName]["vmaxs"])

        if method == "mean":
            climDictionary[varName] = (np.mean(vmins), np.mean(vmaxs))
        elif method == "percentile":
            climDictionary[varName] = (
                np.percentile(vmins, lower_pct),
                np.percentile(vmaxs, upper_pct)
            )
        elif method == "max":
            climDictionary[varName] = (
                np.nanmin(vmins),
                np.nanmax(vmaxs)
            )
        else:
            raise ValueError("method must be 'mean' or 'percentile'")

    return climDictionary, full_climDictionary

def LoadOrCreateCLims(ModelData, varNames,
                              method="max"):
    """
    Loads both the summarized and full clim dictionaries if they exist.
    Otherwise computes, saves, and returns them.
    Returns:
        climDictionary, full_climDictionary
    """
    import pickle
    import os

    
    fileName=f"climDictionary_{ModelData.region}_{ModelData.case}_{ModelData.mpType}_spinup{ModelData_NSSL.spinup_hours}hrs.pkl"
    filePath = os.path.join(outputDirectory,fileName)

    if os.path.exists(filePath):
        print(f"Loading existing clim dictionaries from {filePath}")
        with open(filePath, "rb") as f:
            data = pickle.load(f)

        # backward compatibility: handle old format
        if isinstance(data, tuple):
            climDictionary, full_climDictionary = data
        else:
            climDictionary = data.get("climDictionary", {})
            full_climDictionary = data.get("full_climDictionary", {})
    else:
        print(f"File {filePath} not found. Computing new clim dictionaries.")
        climDictionary, full_climDictionary = GetCLims_Average(
            ModelData, varNames, method=method
        )
        with open(filePath, "wb") as f:
            pickle.dump(
                {"climDictionary": climDictionary,
                 "full_climDictionary": full_climDictionary},
                f
            )
        print(f"Saved clim dictionaries to {filePath}")

    return climDictionary, full_climDictionary

In [ ]:
#PlotVariable_with_Borders()

# Preload map features once
COAST = cfeature.COASTLINE.with_scale("50m")
BORDERS = cfeature.BORDERS.with_scale("50m")
STATES = cfeature.STATES.with_scale("50m")
LAND = cfeature.LAND.with_scale("50m")
LAKES = cfeature.LAKES.with_scale("50m")

# def CreateAxis():
#     # Create figure
#     fig, axis = plt.subplots(
#         subplot_kw={'projection': ccrs.PlateCarree()},
#         figsize=(9, 5))
#     return fig,axis

def CreateAxis():
    """
    Creates a 1×2 subplot using GridSpec with PlateCarree projection.
    Returns the figure and axes.
    """
    fig = plt.figure(figsize=(15, 5))
    gs = fig.add_gridspec(nrows=1, ncols=2, width_ratios=[1, 1], wspace=0.3)

    ax0 = fig.add_subplot(gs[0, 0], projection=ccrs.PlateCarree())
    ax1 = fig.add_subplot(gs[0, 1], projection=ccrs.PlateCarree())
    axes = [ax0, ax1]
    return fig, axes

def PlotVariable_with_Borders(axis, 
                              variable, varName, lat,lon, multiplier, 
                              clim=(None,None), norm=None, cmap="viridis",
                              title=None, units=None,
                              center_colorbar=False, extend=None):
    """
    Plot a uxarray or xarray variable on a map with coastlines, borders, and states,
    using Matplotlib (static PNG output). Works headlessly — no Selenium needed.
    """

    num_levels=19
    if clim != (None,None):
        levels = multiplier*np.linspace(clim[0],clim[1],num_levels)
    else:
        levels=num_levels

    # if center_colorbar==True:
    #     cmap = "RdBu_r"
    #     vmax = max(abs(clim[0]), abs(clim[1]));  vmin = -vmax
    #     norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)
    #     levels = multiplier*np.linspace(vmin,vmax,num_levels)
    if center_colorbar==True:
        cmap = "RdBu_r"
        vmax = clim[1]; vmin = clim[0]
        norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)
        levels = multiplier*np.linspace(vmin,vmax,num_levels)
    
    matrix = multiplier*variable.data
        
    im = axis.contourf(
        lon, lat, matrix,
        levels=levels,
        cmap=cmap,
        norm=norm,
        transform=ccrs.PlateCarree(),
        extend=extend
    ) 

    # Add map features
    axis.add_feature(COAST, linewidth=1)
    axis.add_feature(BORDERS, linewidth=0.8)
    axis.add_feature(STATES, linewidth=0.5)
    axis.add_feature(LAND, facecolor="lightgray", alpha=0.3)
    axis.add_feature(LAKES, edgecolor="k", facecolor="none")

    # Colorbar
    if units is not None:
        label=varName +fr" ({units})"
    else: 
        label=varName

    
    cbar = plt.colorbar(im, ax=axis, orientation="vertical", label=label)    

    #LABELS
    # Set extent to your data range (forces lat/lon ticks)
    axis.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=ccrs.PlateCarree())
    
    # Add lat/lon ticks with degrees
    axis.set_xticks(np.linspace(lon.min(), lon.max(), 5), crs=ccrs.PlateCarree())
    axis.set_yticks(np.linspace(lat.min(), lat.max(), 5), crs=ccrs.PlateCarree())
    
    # # Format tick labels as degrees
    # lon_formatter = ccrs.LongitudeFormatter()
    # lat_formatter = ccrs.LatitudeFormatter()
    # axis.xaxis.set_major_formatter(lon_formatter)
    # axis.yaxis.set_major_formatter(lat_formatter)
    if title is not None:
        axis.set_title(title)
    axis.set_xlabel("Longitude (°E)")
    axis.set_ylabel("Latitude (°N)")
    return axis

def SplitTimeString(timeString):
    date, time = timeString.split('_')
    time = time.replace('.', ':')
    return date,time

In [ ]:
def GetVariableOutputFile_V2(varName, t, ModelData1,ModelData2, outputPlottingDirectory):
    folderName = f"{ModelData1.region}_{ModelData1.case}_{ModelData1.mpType}vs{ModelData2.mpType}_{ModelData1.spinup_hours}hrs/{varName}"
    timeString = ModelData1.timeStrings[t]
    fileName = f"{varName}_{timeString}.png"
    filePath = DirectoryManager.GetOutputFile(outputPlottingDirectory, folderName, fileName)
    return filePath

def MakePlots(ModelData1, ModelData2, 
              variableDictionary1, variableDictionary2,
              save=False):

    for varName in variableDictionary1:
        # Get contents for this variable from both dictionaries
        contents1 = variableDictionary1[varName]
        contents2 = variableDictionary2[varName]
        clim1 = contents1["clim"]; clim2 = contents2["clim"]
        combined_clim = (min(clim1[0], clim2[0]), max(clim1[1], clim2[1]))

        # Create new figure with 1x2 layout
        fig, axes = CreateAxis()

        # Loop over both models
        for contents, ModelData, axis in zip(
            [contents1, contents2],
            [ModelData1, ModelData2],
            axes):

            data = contents["data"]
            lat  = contents["lat"]
            lon  = contents["lon"]
            units = contents["units"]
            multiplier = contents["multiplier"]
            # clim = contents["clim"]
            center_colorbar = contents["center_colorbar"]
            cmap = contents["cmap"]

            # print(data.data);asdf #*

            date, time = SplitTimeString(ModelData.timeStrings[t])
            title = f"{ModelData.region}/{ModelData.case}/{ModelData.mpType} on {date} at {time}"

            PlotVariable_with_Borders(axis,
                                      data, varName, lat, lon, multiplier,
                                      clim=combined_clim,
                                      title=title, units=units,
                                      center_colorbar=center_colorbar,
                                      cmap=cmap)

        if save:
            outputFilePath = GetVariableOutputFile_V2(varName, t, ModelData1,ModelData2, outputPlottingDirectory)
            FigurePlotting_Class.SaveUniformFigure(fig, outputFilePath)

In [ ]:
#################
#RUNNING

In [ ]:
#defining variable names
varNames = ["Theta_rho"]

In [ ]:
climDictionary_NSSL, _ = LoadOrCreateCLims(ModelData_NSSL, varNames)
climDictionary_TEMPO, _ = LoadOrCreateCLims(ModelData_TEMPO, varNames)

In [ ]:
#running
for count, t in enumerate(tqdm(num_elements, desc="Processing timesteps")):
    #Loading Data
    variableSubset_Dictionary_NSSL = {}; variableSubset_Dictionary_TEMPO = {}
    for varName in varNames:
        variable_NSSL = LoadAdditionalVariableData(ModelData_NSSL, varName,t,printout=True)
        variable_TEMPO = LoadAdditionalVariableData(ModelData_TEMPO, varName,t,printout=True)
        variableSubset_NSSL, lat, lon = DataOperator_Class.GetVariable_Subset2(ModelData_NSSL, variable_NSSL)
        variableSubset_TEMPO, _, _ = DataOperator_Class.GetVariable_Subset2(ModelData_TEMPO, variable_TEMPO)

        variableSubset_Dictionary_NSSL[varName] = variableSubset_NSSL
        variableSubset_Dictionary_TEMPO[varName] = variableSubset_TEMPO
    # running
    variableDictionary_NSSL = BuildVariableDictionary(ModelData_NSSL, varNames, variableSubset_Dictionary_NSSL,
                                                      lat,lon,climDictionary_NSSL)
    variableDictionary_TEMPO = BuildVariableDictionary(ModelData_TEMPO, varNames, variableSubset_Dictionary_TEMPO,
                                                       lat,lon,climDictionary_TEMPO)
    MakePlots(ModelData_NSSL, ModelData_TEMPO,
              variableDictionary_NSSL, variableDictionary_TEMPO, save=True)

In [ ]:
#################
#MAKING ANIMATION
ANIMATE=False #keep false when running with bash code
ANIMATE=True

In [ ]:
if ANIMATE==True:
    #Needed Libraries
    # from matplotlib.animation import FuncAnimation, PillowWriter
    # from PIL import Image
    
    # from moviepy import VideoFileClip, vfx
    
    #Importing AnimationPlotting_Class
    sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis","MPAS_Model_Data"))
    from CLASSES_PlottingModelData import AnimationPlotting_Class

In [ ]:
def GetVariableInputFiles(varName, ModelData1,ModelData2, outputPlottingDirectory):
    folderName = f"{ModelData1.region}_{ModelData1.case}_{ModelData1.mpType}vs{ModelData2.mpType}_{ModelData1.spinup_hours}hrs/{varName}"
    fileName = f"{varName}*.png"
    filePattern = DirectoryManager.GetOutputFile(outputPlottingDirectory, folderName, fileName)
    print(filePattern)
    filePaths = DirectoryManager.GetSortedFileListByTimestamp(filePattern)
    return filePaths

def GetPlottingFileName(varName, ModelData1,ModelData2, outputPlottingDirectory, extension="mp4"):
    folderName = f"{ModelData1.region}_{ModelData1.case}_{ModelData1.mpType}vs{ModelData2.mpType}_{ModelData1.spinup_hours}hrs/{varName}"
    plottingFileName = f"{varName}.{extension}"
    
    plottingFilePath = DirectoryManager.GetOutputFile(outputPlottingDirectory, folderName, plottingFileName)
    return plottingFilePath

In [ ]:
# PNGtoMP4 VERSION
if ANIMATE==True:
    
    # running animation
    fps = AnimationPlotting_Class.CalculateFPS(num_frames=ModelData_NSSL.Ntime, time_interval_minutes=15, desired_duration_min=1)
    for varName in varNames:
        if varName in ["greenfrac"]: continue
        print(f"Working on {varName}","\n")
    
        # Setting up output file
        print("getting file information")
        imageFiles = GetVariableInputFiles(varName, ModelData_NSSL,ModelData_TEMPO, outputPlottingDirectory)
        plottingFilePath = GetPlottingFileName(varName, ModelData_NSSL,ModelData_TEMPO, outputPlottingDirectory)

        print("converting PNGs to MP4")
        if varName in ["q2","qfx","cin","rainnc+rainc","refl10cm_1km"]: #these one's have problems plotting for some reason
            resize=True
        else:
            resize=False
        AnimationPlotting_Class.PNGsToMP4(imageFiles, plottingFilePath, fps=fps)